# Backstage: GitHub-Authentifizierung

Die GitHub-Authentifizierung ermöglicht die Anmeldung an Backstage über eine GitHub OAuth App.

Das Auth-Modul ist unabhängig von der GitHub Catalog Discovery. Die Catalog Discovery importiert Repositories, während der Auth Provider Benutzer authentifiziert und einer Backstage `User` Entity zuordnet.


## GitHub Auth Provider installieren

Das GitHub Auth Provider Modul wird im Backstage-Backend installiert.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-github-provider


## Backend-Modul registrieren

Das GitHub Auth Provider Modul wird im Backstage-Backend registriert, damit die GitHub OAuth Endpunkte beim Start bereitgestellt werden.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

grep -q \
  "plugin-auth-backend-module-github-provider" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-auth-backend-module-github-provider'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-auth-backend-module-github-provider


## GitHub OAuth App erstellen

In GitHub:

* Profilbild → **Settings**
* **Developer settings**
* **OAuth Apps**
* **New OAuth App**
* Als Application name beispielsweise `Backstage GitHub Auth` eintragen
* Als Homepage URL die unten ausgegebene Frontend-URL eintragen
* Als Authorization callback URL die unten ausgegebene Callback-URL eintragen
* OAuth App registrieren
* `Client ID` kopieren
* Ein neues `Client Secret` erzeugen und sofort kopieren
* Werte in [env-platen.py](../../data/env-platen.py) als `AUTH_GITHUB_CLIENT_ID` und `AUTH_GITHUB_CLIENT_SECRET` eintragen

Die Callback-URL muss exakt auf den Backstage Auth Handler zeigen und darf nach `frame` keinen abschliessenden Slash enthalten.


In [ ]:
%%bash
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_PORT="3001"

echo "GitHub Homepage URL:"
echo "http://${BACKSTAGE_HOSTNAME}:${BACKSTAGE_PORT}"
echo
echo "GitHub Authorization callback URL:"
echo "http://localhost:7007/api/auth/github/handler/frame"


## Umgebungsvariablen prüfen

Die OAuth-Zugangsdaten werden aus `env-platen.py` geladen. Das Script zeigt nur, ob die Variablen gesetzt sind, und gibt keine Secrets aus.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
set +a

test -n "${GITHUB_CLIENT_ID}" \
  && echo "GITHUB_CLIENT_ID ist gesetzt" \
  || echo "GITHUB_CLIENT_ID fehlt"

test -n "${GITHUB_SECRET}" \
  && echo "GITHUB_SECRET ist gesetzt" \
  || echo "GITHUB_SECRET fehlt"


## GitHub Auth Provider konfigurieren

Der GitHub Provider wird in einer separaten Backstage-Konfigurationsdatei eingerichtet.

Der Resolver `usernameMatchingUserEntityName` ordnet den GitHub-Benutzernamen einer Backstage `User` Entity mit demselben `metadata.name` zu.

Beispiel: Der GitHub-Benutzer `marcel-cli` benötigt im Backstage Catalog eine `User` Entity mit `metadata.name: marcel-cli`.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

cat > app-config.github-auth.yaml <<'EOF'
auth:
  environment: development
  providers:
    github:
      development:
        clientId: ${GITHUB_CLIENT_ID}
        clientSecret: ${GITHUB_SECRET}
        signIn:
          resolvers:
            - resolver: usernameMatchingUserEntityName
EOF


## GitHub Login im Frontend konfigurieren

Die neue Backstage Frontend-Architektur verwendet eine `SignInPageBlueprint` Extension.

Das folgende Script erstellt die Extension in `packages/app/src/extensions/githubSignInPage.tsx`. Die Extension wird anschliessend in der vorhandenen `features`-Liste von `packages/app/src/App.tsx` registriert.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

mkdir -p packages/app/src/extensions

cat > packages/app/src/extensions/githubSignInPage.tsx <<'EOF'
import { SignInPage } from '@backstage/core-components';
import { githubAuthApiRef } from '@backstage/core-plugin-api';
import { SignInPageBlueprint } from '@backstage/plugin-app-react';

export const githubSignInPage = SignInPageBlueprint.make({
  params: {
    loader: async () => props => (
      <SignInPage
        {...props}
        provider={{
          id: 'github-auth-provider',
          title: 'GitHub',
          message: 'Mit GitHub anmelden',
          apiRef: githubAuthApiRef,
        }}
      />
    ),
  },
});
EOF

echo "Extension erstellt:"
echo "packages/app/src/extensions/githubSignInPage.tsx"


## Frontend Extension registrieren

Das Script ergänzt den Import und fügt `githubSignInPage` am Anfang der vorhandenen `features`-Liste ein.

Vor der Änderung wird eine Sicherung von `App.tsx` erstellt. Falls die erwartete `features`-Liste nicht gefunden wird, bricht das Script ohne Änderung ab.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

python3 - <<'PY'
from pathlib import Path
import shutil

path = Path("packages/app/src/App.tsx")
backup = path.with_suffix(".tsx.github-auth.bak")
text = path.read_text(encoding="utf-8")

import_line = "import { githubSignInPage } from './extensions/githubSignInPage';"

if import_line not in text:
    lines = text.splitlines()
    last_import = max(
        (index for index, line in enumerate(lines) if line.startswith("import ")),
        default=-1,
    )
    lines.insert(last_import + 1, import_line)
    text = "\n".join(lines) + "\n"

if "githubSignInPage," not in text:
    marker = "features: ["
    if marker not in text:
        raise SystemExit(
            "Keine features-Liste gefunden. App.tsx wurde nicht verändert."
        )
    text = text.replace(marker, marker + "\n    githubSignInPage,", 1)

if not backup.exists():
    shutil.copy2(path, backup)

path.write_text(text, encoding="utf-8")
print(f"App.tsx aktualisiert. Sicherung: {backup}")
PY

grep -n "githubSignInPage" packages/app/src/App.tsx


## Backstage User Entity prüfen

Der gewählte Resolver benötigt eine passende `User` Entity im Software Catalog.

Die bestehende User-Konfiguration kann mit dem folgenden Befehl gesucht werden.


In [ ]:
%%bash
cd ~/mybackstage/

grep -R --line-number --include='*.yaml' --include='*.yml' \
  -E '^kind:[[:space:]]*User|^[[:space:]]*name:[[:space:]]*marcel-cli' \
  examples catalog 2>/dev/null || true


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage GitHub Auth"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn backstage-cli config:print \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.github.yaml \
  --config ~/mybackstage/app-config.github-auth.yaml


## Backstage starten

Der Start-Endpunkt des GitHub Providers muss mit einem HTTP Redirect antworten. 


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage GitHub Auth"
export BACKSTAGE_PORT="3001"

echo "Frontend: http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
echo "Backend:  http://localhost:7007/api/auth/github/start?env=development"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn start \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.github.yaml \
  --config ~/mybackstage/app-config.github-auth.yaml \
  2>&1 | tee /tmp/backstage-github-auth.log


**Links**

- [GitHub Authentication Provider](https://backstage.io/docs/auth/github/provider/)
- [Authentication in Backstage](https://backstage.io/docs/auth/)
- [Sign-in Identities and Resolvers](https://backstage.io/docs/auth/identity-resolver/)
